# KITTI SLAM Comparison Notebook

This notebook reads the current optimized 300-frame results for the ORB / DSO / SVO Python research baselines. It is intentionally tied to the latest result directories:

- `results/final_seq00_300_optimized_f800_p800/benchmark_seq00.json`
- `results/final_seq00_300_optimized_f800_p800/dso_diagnostics_seq00.json`
- `results/dso_ablation_300_optimized_p800/dso_ablation_seq00.json`

Important boundary: these are reproducible Python research baselines, not official paper-level reproductions of ORB-SLAM2, DSO, or SVO.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

try:
    import pandas as pd
except ImportError:
    pd = None

ROOT = Path.cwd()
RESULTS = ROOT / 'results'
BENCHMARK_DIR = RESULTS / 'final_seq00_300_optimized_f800_p800'
ABLATION_DIR = RESULTS / 'dso_ablation_300_optimized_p800'
BENCHMARK_PATH = BENCHMARK_DIR / 'benchmark_seq00.json'
DIAGNOSTICS_PATH = BENCHMARK_DIR / 'dso_diagnostics_seq00.json'
ABLATION_PATH = ABLATION_DIR / 'dso_ablation_seq00.json'

for path in [BENCHMARK_PATH, DIAGNOSTICS_PATH, ABLATION_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

with BENCHMARK_PATH.open('r', encoding='utf-8') as f:
    benchmark = json.load(f)
with DIAGNOSTICS_PATH.open('r', encoding='utf-8') as f:
    diagnostics = json.load(f)
with ABLATION_PATH.open('r', encoding='utf-8') as f:
    ablation = json.load(f)

print('Loaded benchmark:', BENCHMARK_PATH)
print('Loaded DSO diagnostics:', DIAGNOSTICS_PATH)
print('Loaded DSO ablation:', ABLATION_PATH)

## Benchmark Summary

The table below extracts the main 300-frame metrics from the latest optimized benchmark JSON.

In [ ]:
def metric(row, *keys, default=None):
    value = row
    for key in keys:
        if not isinstance(value, dict) or key not in value:
            return default
        value = value[key]
    return value

rows = []
for name in ['orb', 'dso', 'svo']:
    item = benchmark['results'][name]
    rows.append({
        'algorithm': name.upper(),
        'frames': item.get('num_poses'),
        'ate_rmse_m': metric(item, 'ate', 'rmse'),
        'rpe_trans_percent': metric(item, 'rpe', 'translation_percent_mean'),
        'kitti_mean_trans_percent': metric(item, 'kitti_segments', 'translation_percent_mean'),
        'failures': item.get('tracking_failures'),
        'fallbacks': item.get('fallbacks'),
        'loop_candidates': item.get('loop_candidates'),
        'loop_closures': item.get('loop_closures'),
        'avg_runtime_ms': metric(item, 'runtime', 'avg_ms'),
    })

if pd is not None:
    benchmark_df = pd.DataFrame(rows)
    display(benchmark_df)
else:
    for row in rows:
        print(row)

## DSO Diagnostics

The optimized DSO implementation records strict loop semantics and tracking gates. In particular, `loop_closures` is no longer a synonym for loop candidates.

In [ ]:
dso = benchmark['results']['dso']
summary = {
    'tracking_failures': dso.get('tracking_failures'),
    'fallbacks': dso.get('fallbacks'),
    'loop_candidates': dso.get('loop_candidates'),
    'loop_verified': dso.get('loop_verified'),
    'loop_corrections_applied': dso.get('loop_corrections_applied'),
    'loop_closures': dso.get('loop_closures'),
    'ba_runs': dso.get('ba_runs'),
    'ba_accepted': dso.get('ba_accepted'),
    'ba_rejected': dso.get('ba_rejected'),
    'active_points_culled': dso.get('active_points_culled'),
    'reinitializations': dso.get('reinitializations'),
}
summary

In [ ]:
diag_rows = diagnostics if isinstance(diagnostics, list) else diagnostics.get('diagnostics', [])
if pd is not None:
    diag_df = pd.DataFrame(diag_rows)
    keep_cols = [c for c in [
        'frame_id', 'tracking_status', 'failure_reason', 'fallback_used',
        'valid_projected_ratio', 'inlier_ratio', 'residual_p95',
        'cost_ratio', 'consecutive_failures', 'ba_accepted',
        'loop_candidates', 'loop_verified', 'loop_corrections_applied'
    ] if c in diag_df.columns]
    display(diag_df[keep_cols].tail(20))
else:
    print('diagnostic frames:', len(diag_rows))

## Result Figures

In [ ]:
from IPython.display import Image, display

for image_path in [
    BENCHMARK_DIR / 'benchmark_trajectory_seq00.png',
    BENCHMARK_DIR / 'dso_diagnostics_seq00.png',
    ABLATION_DIR / 'dso_ablation_seq00.png',
]:
    if image_path.exists():
        print(image_path)
        display(Image(filename=str(image_path)))
    else:
        print('Missing:', image_path)

## DSO Ablation

Use `strict_full` as the current default DSO. The most important sanity check is that `no_joint_ba` degrades heavily, while `loop_candidates_only` does not inflate true loop closures.

In [ ]:
abl_rows = []
for item in ablation.get('results', []):
    abl_rows.append({
        'config': item.get('config'),
        'ate_rmse_m': metric(item, 'ate', 'rmse'),
        'rpe_trans_percent': metric(item, 'rpe', 'translation_percent_mean'),
        'failures': item.get('tracking_failures'),
        'fallbacks': item.get('fallbacks'),
        'loop_candidates': item.get('loop_candidates'),
        'loop_closures': item.get('loop_closures'),
        'ba_accepted': item.get('ba_accepted'),
        'ba_runs': item.get('ba_runs'),
        'active_points_culled': item.get('active_points_culled'),
    })

if pd is not None:
    ablation_df = pd.DataFrame(abl_rows)
    display(ablation_df.sort_values('ate_rmse_m'))
else:
    for row in sorted(abl_rows, key=lambda x: x['ate_rmse_m'] if x['ate_rmse_m'] is not None else float('inf')):
        print(row)

In [ ]:
if pd is not None:
    plot_df = ablation_df.sort_values('ate_rmse_m')
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(plot_df['config'], plot_df['ate_rmse_m'])
    ax.set_ylabel('ATE RMSE (m)')
    ax.set_title('DSO 300-frame ablation')
    ax.tick_params(axis='x', rotation=45)
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
else:
    print('Install pandas for the compact plotting table.')

## Reproduction Commands

In [ ]:
print(r'''.\.venv\Scripts\python.exe -B run_benchmark.py --data-dir data/kitti_odometry --seq 00 --max-frames 300 --algorithms orb,dso,svo --features 800 --points 800 --svo-points 1500 --output-dir results\final_seq00_300_optimized_f800_p800
.\.venv\Scripts\python.exe -B dso_ablation.py --data-dir data/kitti_odometry --seq 00 --max-frames 300 --points 800 --output-dir results\dso_ablation_300_optimized_p800''')

## Interpretation Notes

- The latest DSO result is not guaranteed to minimize ATE against every ablation; the goal of the six enhancements is stricter acceptance, fairer loop accounting, and better diagnostics.
- `loop_candidates`, `loop_verified`, `loop_corrections_applied`, and `loop_closures` should be discussed separately.
- Old 20/100-frame and pre-optimization outputs are archived under `results/Old/`; they are useful for history, not for the current main table.